# Problema XOR con Redes Neuronales en Scikit-Learn

## ¿Por qué un Perceptrón Simple no puede resolver XOR?

El problema **XOR (OR exclusivo)** es un ejemplo clásico de un problema **no linealmente separable**. 

| X1 | X2 | XOR |
|----|----|-----|
| 0  | 0  | 0   |
| 0  | 1  | 1   |
| 1  | 0  | 1   |
| 1  | 1  | 0   |

Un **perceptrón simple** (sin capas ocultas) solo puede trazar una línea recta para separar las clases. Como los puntos de XOR no se pueden separar con una sola línea, el perceptrón falla.

**Solución**: Añadir una **capa oculta con al menos 2 neuronas** permite a la red crear fronteras de decisión no lineales combinando múltiples líneas.

In [ ]:
# Importar librerías necesarias
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import Perceptron
from matplotlib.colors import ListedColormap

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Crear el Dataset XOR

In [ ]:
# Datos del problema XOR
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

y = np.array([0, 1, 1, 0])  # XOR: devuelve 1 si los inputs son diferentes

print("Datos de entrada (X):")
print(X)
print("\nEtiquetas XOR (y):")
print(y)

## 2. Visualizar el Problema XOR

Observa cómo los puntos de clase 0 (azul) y clase 1 (rojo) no pueden separarse con una sola línea recta.

In [ ]:
# Visualizar los puntos XOR
plt.figure(figsize=(8, 6))
colors = ['blue', 'red']
markers = ['o', 's']

for i, label in enumerate([0, 1]):
    mask = y == label
    plt.scatter(X[mask, 0], X[mask, 1], 
                c=colors[i], marker=markers[i], 
                s=200, edgecolor='black', linewidth=2,
                label=f'Clase {label}')

plt.xlabel('X1', fontsize=12)
plt.ylabel('X2', fontsize=12)
plt.title('Problema XOR - No Linealmente Separable', fontsize=14)
plt.legend(fontsize=11)
plt.xlim(-0.5, 1.5)
plt.ylim(-0.5, 1.5)
plt.grid(True, alpha=0.3)

# Mostrar que no hay línea que separe las clases
plt.annotate('¿Puedes trazar UNA línea\nque separe rojos de azules?', 
             xy=(0.5, 0.5), fontsize=10, ha='center',
             bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))
plt.tight_layout()
plt.show()

## 3. Intento con Perceptrón Simple (Sin Capa Oculta)

El perceptrón simple no tiene capas ocultas. Veremos que **no puede aprender** el patrón XOR.

In [ ]:
# Entrenar un Perceptrón Simple
perceptron = Perceptron(max_iter=1000, random_state=42)
perceptron.fit(X, y)

# Predicciones
y_pred_perceptron = perceptron.predict(X)
accuracy_perceptron = (y_pred_perceptron == y).mean() * 100

print("=" * 50)
print("PERCEPTRÓN SIMPLE (Sin capa oculta)")
print("=" * 50)
print(f"\nPredicciones: {y_pred_perceptron}")
print(f"Valores reales: {y}")
print(f"\nPrecisión: {accuracy_perceptron:.0f}%")
print("\n⚠️ El perceptrón NO puede resolver XOR (máximo 50-75% de precisión)")

## 4. Red Neuronal con 2 Neuronas Ocultas (MLP)

Ahora usamos un **MLPClassifier** con una capa oculta de **2 neuronas**. Esta arquitectura sí puede resolver XOR.

```
Arquitectura:
    [Entrada]      [Capa Oculta]     [Salida]
       X1  ─────┐
                ├──── N1 ────┐
       X2  ─────┤            ├──── Output
                ├──── N2 ────┘
```

In [ ]:
# Red Neuronal con 2 neuronas ocultas
# Nota: random_state=42 no converge con 2 neuronas, usamos random_state=1
mlp = MLPClassifier(
    hidden_layer_sizes=(2,),  # UNA capa oculta con 2 neuronas
    activation='tanh',         # Función de activación no lineal
    solver='lbfgs',            # Optimizador eficiente para datasets pequeños
    max_iter=1000,
    random_state=1             # Semilla que permite convergencia
)

mlp.fit(X, y)

# Predicciones
y_pred_mlp = mlp.predict(X)
accuracy_mlp = (y_pred_mlp == y).mean() * 100

print("=" * 50)
print("RED NEURONAL CON 2 NEURONAS OCULTAS")
print("=" * 50)
print(f"\nArquitectura: {mlp.hidden_layer_sizes}")
print(f"Función de activación: {mlp.activation}")
print(f"\nPredicciones: {y_pred_mlp}")
print(f"Valores reales: {y}")
print(f"\nPrecisión: {accuracy_mlp:.0f}%")
print("\n✅ ¡La red neuronal SÍ puede resolver XOR!")

### ⚠️ ¿Por qué la convergencia depende de `random_state`?

La semilla aleatoria determina los **pesos iniciales** de la red. El problema es:

1. **Superficie de pérdida no convexa**: La función de coste tiene múltiples mínimos locales
2. **Punto de partida crítico**: Con solo 2 neuronas, el espacio de soluciones es limitado
3. **El optimizador puede quedarse "atrapado"** en un mínimo local que no resuelve XOR

Veamos qué semillas funcionan y cuáles no:

In [ ]:
# Probar múltiples semillas para ver cuáles convergen
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("ANÁLISIS DE CONVERGENCIA SEGÚN LA SEMILLA (random_state)")
print("=" * 60)
print("\nProbando semillas del 0 al 49 con 2 neuronas ocultas...\n")

convergen = []
no_convergen = []

for seed in range(50):
    mlp_test = MLPClassifier(
        hidden_layer_sizes=(2,),
        activation='tanh',
        solver='lbfgs',
        max_iter=1000,
        random_state=seed
    )
    mlp_test.fit(X, y)
    acc = (mlp_test.predict(X) == y).mean() * 100
    
    if acc == 100:
        convergen.append(seed)
    else:
        no_convergen.append((seed, acc))

print(f"✅ Semillas que CONVERGEN (100%): {len(convergen)}")
print(f"   {convergen}")

print(f"\n❌ Semillas que NO convergen: {len(no_convergen)}")
print(f"   Semilla → Precisión")
for seed, acc in no_convergen[:10]:  # Mostrar solo las primeras 10
    print(f"   {seed:3d}    → {acc:.0f}%")
if len(no_convergen) > 10:
    print(f"   ... y {len(no_convergen) - 10} más")

print(f"\n📊 Tasa de éxito: {len(convergen)/50*100:.0f}% de las semillas convergen")
print("\n💡 Conclusión: La inicialización de pesos es CRUCIAL")
print("   Con 2 neuronas, muchas semillas llevan a mínimos locales malos.")

In [ ]:
# Visualizar cómo aumentar neuronas mejora la tasa de convergencia
neuronas_a_probar = [2, 3, 4, 5, 10]
resultados = {}

for n_neuronas in neuronas_a_probar:
    exitos = 0
    for seed in range(50):
        mlp_test = MLPClassifier(
            hidden_layer_sizes=(n_neuronas,),
            activation='tanh',
            solver='lbfgs',
            max_iter=1000,
            random_state=seed
        )
        mlp_test.fit(X, y)
        if (mlp_test.predict(X) == y).mean() == 1.0:
            exitos += 1
    resultados[n_neuronas] = exitos / 50 * 100

# Gráfico
plt.figure(figsize=(10, 5))
bars = plt.bar(resultados.keys(), resultados.values(), color=['red' if v < 80 else 'orange' if v < 95 else 'green' for v in resultados.values()], edgecolor='black')
plt.axhline(y=100, color='green', linestyle='--', alpha=0.5, label='100% convergencia')
plt.xlabel('Número de Neuronas Ocultas', fontsize=12)
plt.ylabel('% de Semillas que Convergen', fontsize=12)
plt.title('Más neuronas ocultas = Mayor probabilidad de convergencia\n(Probando 50 semillas diferentes)', fontsize=13)
plt.ylim(0, 105)

# Añadir valores sobre las barras
for bar, (n, v) in zip(bars, resultados.items()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{v:.0f}%', 
             ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Con más neuronas, la red tiene más flexibilidad y es más fácil encontrar una solución.")

### 🎯 Visualización 3D de la Superficie de Decisión

La red neuronal aprende una **superficie de decisión** en el espacio. 
- El eje Z representa la **probabilidad de clase 1**
- El plano en Z=0.5 es la **frontera de decisión**
- Los puntos por encima se clasifican como clase 1, los de abajo como clase 0

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Crear malla de puntos para la superficie
resolution = 50
x1_range = np.linspace(-0.2, 1.2, resolution)
x2_range = np.linspace(-0.2, 1.2, resolution)
X1_grid, X2_grid = np.meshgrid(x1_range, x2_range)

# Obtener probabilidades de la red neuronal entrenada
X_grid_flat = np.c_[X1_grid.ravel(), X2_grid.ravel()]
Z_proba = mlp.predict_proba(X_grid_flat)[:, 1].reshape(X1_grid.shape)

# Crear figura 3D
fig = plt.figure(figsize=(16, 6))

# --- Gráfico 1: Superficie de probabilidad ---
ax1 = fig.add_subplot(1, 2, 1, projection='3d')

# Superficie de probabilidad
surf = ax1.plot_surface(X1_grid, X2_grid, Z_proba, cmap='coolwarm', 
                         alpha=0.8, edgecolor='none')

# Plano de decisión en Z=0.5
ax1.plot_surface(X1_grid, X2_grid, np.ones_like(X1_grid) * 0.5, 
                  alpha=0.3, color='green', label='Frontera (Z=0.5)')

# Puntos XOR con su altura real (probabilidad)
proba_puntos = mlp.predict_proba(X)[:, 1]
colors_3d = ['blue' if yi == 0 else 'red' for yi in y]
ax1.scatter(X[:, 0], X[:, 1], proba_puntos, c=colors_3d, s=200, 
            edgecolor='black', linewidth=2, zorder=5)

# Líneas verticales para ver la altura
for i in range(len(X)):
    ax1.plot([X[i, 0], X[i, 0]], [X[i, 1], X[i, 1]], [0, proba_puntos[i]], 
             color='gray', linestyle='--', alpha=0.5)

ax1.set_xlabel('X1', fontsize=11)
ax1.set_ylabel('X2', fontsize=11)
ax1.set_zlabel('P(Clase 1)', fontsize=11)
ax1.set_title('Superficie de Decisión de la Red Neuronal\n(La curva en 3D que resuelve XOR)', fontsize=12)
ax1.view_init(elev=25, azim=45)
fig.colorbar(surf, ax=ax1, shrink=0.5, label='Probabilidad Clase 1')

# --- Gráfico 2: Vista superior (contorno) con proyección ---
ax2 = fig.add_subplot(1, 2, 2)

# Contorno de probabilidad
contour = ax2.contourf(X1_grid, X2_grid, Z_proba, levels=20, cmap='coolwarm', alpha=0.8)
ax2.contour(X1_grid, X2_grid, Z_proba, levels=[0.5], colors='green', linewidths=3)

# Puntos XOR
for i, label in enumerate([0, 1]):
    mask = y == label
    ax2.scatter(X[mask, 0], X[mask, 1], c=['blue', 'red'][i], 
                s=200, edgecolor='black', linewidth=2, label=f'Clase {label}')

ax2.set_xlabel('X1', fontsize=11)
ax2.set_ylabel('X2', fontsize=11)
ax2.set_title('Vista Superior (Contorno)\nLínea verde = Frontera de decisión', fontsize=12)
ax2.legend()
fig.colorbar(contour, ax=ax2, label='Probabilidad Clase 1')

plt.tight_layout()
plt.show()

print("\n📊 Interpretación:")
print("   - La superficie 3D es la 'montaña' que la red aprende")
print("   - Los puntos azules (clase 0) quedan en los 'valles' (baja probabilidad)")
print("   - Los puntos rojos (clase 1) quedan en las 'cimas' (alta probabilidad)")
print("   - El plano verde en Z=0.5 corta la superficie creando la frontera curva")

### 🧠 Visualización de lo que hace CADA neurona oculta

Cada neurona oculta aprende un **hiperplano** (una línea en 2D). La combinación de las dos líneas crea la frontera no lineal:

In [ ]:
# Visualizar lo que aprende cada neurona oculta
W1 = mlp.coefs_[0]    # Pesos entrada -> oculta (2x2)
b1 = mlp.intercepts_[0]  # Bias capa oculta (2,)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Calcular la activación de cada neurona para toda la malla
Z_neuron1 = np.tanh(X1_grid * W1[0, 0] + X2_grid * W1[1, 0] + b1[0])
Z_neuron2 = np.tanh(X1_grid * W1[0, 1] + X2_grid * W1[1, 1] + b1[1])

# --- Neurona 1 ---
im1 = axes[0].contourf(X1_grid, X2_grid, Z_neuron1, levels=20, cmap='RdBu_r', alpha=0.8)
axes[0].contour(X1_grid, X2_grid, Z_neuron1, levels=[0], colors='black', linewidths=3)
for i, label in enumerate([0, 1]):
    mask = y == label
    axes[0].scatter(X[mask, 0], X[mask, 1], c=['blue', 'red'][i], 
                    s=150, edgecolor='black', linewidth=2)
axes[0].set_xlabel('X1')
axes[0].set_ylabel('X2')
axes[0].set_title(f'Neurona Oculta 1\n(w=[{W1[0,0]:.2f}, {W1[1,0]:.2f}], b={b1[0]:.2f})', fontsize=11)
fig.colorbar(im1, ax=axes[0], label='Activación')

# --- Neurona 2 ---
im2 = axes[1].contourf(X1_grid, X2_grid, Z_neuron2, levels=20, cmap='RdBu_r', alpha=0.8)
axes[1].contour(X1_grid, X2_grid, Z_neuron2, levels=[0], colors='black', linewidths=3)
for i, label in enumerate([0, 1]):
    mask = y == label
    axes[1].scatter(X[mask, 0], X[mask, 1], c=['blue', 'red'][i], 
                    s=150, edgecolor='black', linewidth=2)
axes[1].set_xlabel('X1')
axes[1].set_ylabel('X2')
axes[1].set_title(f'Neurona Oculta 2\n(w=[{W1[0,1]:.2f}, {W1[1,1]:.2f}], b={b1[1]:.2f})', fontsize=11)
fig.colorbar(im2, ax=axes[1], label='Activación')

# --- Combinación final ---
im3 = axes[2].contourf(X1_grid, X2_grid, Z_proba, levels=20, cmap='coolwarm', alpha=0.8)
axes[2].contour(X1_grid, X2_grid, Z_proba, levels=[0.5], colors='green', linewidths=3)
# Dibujar las líneas de las neuronas individuales
axes[2].contour(X1_grid, X2_grid, Z_neuron1, levels=[0], colors='black', linewidths=2, linestyles='--', alpha=0.7)
axes[2].contour(X1_grid, X2_grid, Z_neuron2, levels=[0], colors='black', linewidths=2, linestyles='--', alpha=0.7)
for i, label in enumerate([0, 1]):
    mask = y == label
    axes[2].scatter(X[mask, 0], X[mask, 1], c=['blue', 'red'][i], 
                    s=150, edgecolor='black', linewidth=2)
axes[2].set_xlabel('X1')
axes[2].set_ylabel('X2')
axes[2].set_title('Combinación: 2 Líneas → Frontera Curva\n(líneas negras = cada neurona)', fontsize=11)
fig.colorbar(im3, ax=axes[2], label='Prob. Clase 1')

plt.suptitle('Cómo 2 neuronas crean una frontera no lineal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n🔑 CLAVE:")
print("   • Cada neurona oculta traza UNA línea (hiperplano en el espacio de entrada)")
print("   • La línea negra en cada gráfico es donde la neurona cambia de signo")
print("   • La capa de salida COMBINA las dos líneas")
print("   • Resultado: una frontera curva que separa correctamente XOR")

## 5. Comparación Visual: Frontera de Decisión

Veamos cómo cada modelo intenta separar las clases:

In [ ]:
def plot_decision_boundary(model, X, y, ax, title):
    """Dibuja la frontera de decisión de un clasificador"""
    # Crear una malla de puntos
    h = 0.02  # Paso de la malla
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predecir para cada punto de la malla
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Colores para las regiones
    cmap_light = ListedColormap(['#AAAAFF', '#FFAAAA'])
    cmap_bold = ListedColormap(['#0000FF', '#FF0000'])
    
    # Dibujar regiones de decisión
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_light)
    ax.contour(xx, yy, Z, colors='black', linewidths=2, linestyles='--')
    
    # Dibujar puntos de datos
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold, 
                         s=200, edgecolor='black', linewidth=2)
    
    ax.set_xlabel('X1', fontsize=11)
    ax.set_ylabel('X2', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    
    return scatter

# Crear figura comparativa
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Perceptrón Simple
plot_decision_boundary(perceptron, X, y, axes[0], 
                       f'Perceptrón Simple\n(Precisión: {accuracy_perceptron:.0f}%)')
axes[0].annotate('❌ Solo puede trazar\nUNA línea recta', 
                 xy=(0.5, -0.3), fontsize=10, ha='center',
                 bbox=dict(boxstyle='round', facecolor='lightyellow'))

# MLP con 2 neuronas ocultas
plot_decision_boundary(mlp, X, y, axes[1], 
                       f'MLP con 2 Neuronas Ocultas\n(Precisión: {accuracy_mlp:.0f}%)')
axes[1].annotate('✅ Puede crear fronteras\nNO lineales', 
                 xy=(0.5, -0.3), fontsize=10, ha='center',
                 bbox=dict(boxstyle='round', facecolor='lightgreen'))

plt.suptitle('Comparación: Perceptrón Simple vs Red Neuronal', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. ¿Cómo lo hace? Transformación del Espacio

La clave está en que las **2 neuronas ocultas transforman el espacio de entrada** a un nuevo espacio donde los datos SÍ son linealmente separables.

Veamos qué hacen las neuronas ocultas:

In [ ]:
# Obtener la activación de la capa oculta
# Calcular manualmente la salida de la capa oculta
def get_hidden_activations(mlp, X):
    """Calcula las activaciones de la capa oculta"""
    W1 = mlp.coefs_[0]    # Pesos entrada -> oculta
    b1 = mlp.intercepts_[0]  # Bias de capa oculta
    
    # Cálculo de la capa oculta: activación(X @ W1 + b1)
    z1 = X @ W1 + b1
    if mlp.activation == 'tanh':
        h1 = np.tanh(z1)
    elif mlp.activation == 'relu':
        h1 = np.maximum(0, z1)
    else:
        h1 = 1 / (1 + np.exp(-z1))  # sigmoid
    return h1

# Obtener activaciones
hidden_activations = get_hidden_activations(mlp, X)

# Visualizar transformación
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Espacio original
cmap_bold = ListedColormap(['#0000FF', '#FF0000'])
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold, 
                s=200, edgecolor='black', linewidth=2)
axes[0].set_xlabel('X1', fontsize=11)
axes[0].set_ylabel('X2', fontsize=11)
axes[0].set_title('1. Espacio Original\n(NO separable linealmente)', fontsize=12)
for i, (x, y_val) in enumerate(zip(X, y)):
    axes[0].annotate(f'({x[0]},{x[1]})→{y_val}', xy=(x[0]+0.05, x[1]+0.1), fontsize=9)
axes[0].set_xlim(-0.3, 1.5)
axes[0].set_ylim(-0.3, 1.5)

# 2. Espacio transformado por las neuronas ocultas
axes[1].scatter(hidden_activations[:, 0], hidden_activations[:, 1], 
                c=y, cmap=cmap_bold, s=200, edgecolor='black', linewidth=2)
axes[1].set_xlabel('Neurona Oculta 1', fontsize=11)
axes[1].set_ylabel('Neurona Oculta 2', fontsize=11)
axes[1].set_title('2. Espacio Transformado\n(Salida de capa oculta)', fontsize=12)

# Añadir línea de separación en el espacio transformado
# Encontrar una línea que separe
h_min, h_max = hidden_activations.min() - 0.2, hidden_activations.max() + 0.2
axes[1].plot([h_min, h_max], [h_max, h_min], 'g--', linewidth=2, label='Frontera lineal')
axes[1].legend()

# 3. Diagrama conceptual
axes[2].axis('off')
axes[2].set_title('3. Arquitectura de la Red', fontsize=12)

# Dibujar la arquitectura
# Posiciones de las neuronas
input_y = [0.3, 0.7]
hidden_y = [0.3, 0.7]
output_y = [0.5]

# Neuronas de entrada
for i, ypos in enumerate(input_y):
    circle = plt.Circle((0.2, ypos), 0.08, color='lightblue', ec='black', lw=2)
    axes[2].add_patch(circle)
    axes[2].text(0.2, ypos, f'X{i+1}', ha='center', va='center', fontsize=10, fontweight='bold')

# Neuronas ocultas
for i, ypos in enumerate(hidden_y):
    circle = plt.Circle((0.5, ypos), 0.08, color='lightgreen', ec='black', lw=2)
    axes[2].add_patch(circle)
    axes[2].text(0.5, ypos, f'H{i+1}', ha='center', va='center', fontsize=10, fontweight='bold')

# Neurona de salida
circle = plt.Circle((0.8, 0.5), 0.08, color='lightyellow', ec='black', lw=2)
axes[2].add_patch(circle)
axes[2].text(0.8, 0.5, 'Out', ha='center', va='center', fontsize=10, fontweight='bold')

# Conexiones
for iy in input_y:
    for hy in hidden_y:
        axes[2].annotate('', xy=(0.42, hy), xytext=(0.28, iy),
                        arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

for hy in hidden_y:
    axes[2].annotate('', xy=(0.72, 0.5), xytext=(0.58, hy),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# Etiquetas
axes[2].text(0.2, 0.0, 'Capa\nEntrada', ha='center', fontsize=9)
axes[2].text(0.5, 0.0, 'Capa Oculta\n(2 neuronas)', ha='center', fontsize=9)
axes[2].text(0.8, 0.0, 'Capa\nSalida', ha='center', fontsize=9)

axes[2].set_xlim(0, 1)
axes[2].set_ylim(-0.15, 1)

plt.suptitle('Cómo la Red Neuronal Transforma el Problema XOR', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Resumen y Conclusión

### ¿Por qué el Perceptrón Simple falla con XOR?
- Solo puede crear **una frontera de decisión lineal** (una línea recta)
- XOR requiere separar puntos que NO están alineados linealmente

### ¿Por qué 2 neuronas ocultas son suficientes?
1. **Cada neurona oculta aprende una función lineal diferente**
2. **La combinación de ambas** crea una frontera no lineal
3. El espacio se **transforma** para que los datos sean separables

### Analogía Visual
Imagina que tienes 4 puntos en una hoja de papel que no puedes separar con una línea recta. 
Pero si **doblas el papel** (transformación), los puntos quedan en diferentes altura y ahora SÍ puedes separarlos con un plano.

**Eso es exactamente lo que hace la capa oculta: "dobla" el espacio de entrada.**

In [ ]:
# Mostrar los pesos aprendidos por la red
print("=" * 60)
print("PESOS APRENDIDOS POR LA RED NEURONAL")
print("=" * 60)

print("\n📊 Pesos de Entrada → Capa Oculta:")
print(f"   Forma: {mlp.coefs_[0].shape} (2 entradas × 2 neuronas ocultas)")
print(f"   Valores:\n{mlp.coefs_[0]}")

print("\n📊 Bias de Capa Oculta:")
print(f"   Forma: {mlp.intercepts_[0].shape}")
print(f"   Valores: {mlp.intercepts_[0]}")

print("\n📊 Pesos de Capa Oculta → Salida:")
print(f"   Forma: {mlp.coefs_[1].shape} (2 neuronas ocultas × 1 salida)")
print(f"   Valores:\n{mlp.coefs_[1]}")

print("\n📊 Bias de Salida:")
print(f"   Valores: {mlp.intercepts_[1]}")

print("\n" + "=" * 60)
print("✅ Con estos pesos, la red puede predecir XOR correctamente!")
print("=" * 60)